# Manhwa Studio — Renderizador CPU no Google Colab
Este notebook conecta o site ao FFmpeg executado na sessão do Google Colab. Conecte o ambiente e execute as células em ordem. Copie a URL HTTPS e o token gerados para a área **Conexão Google Colab** do site.

Use apenas vídeos e imagens que você tem direito de processar. O Colab gratuito pode encerrar sessões e não garante renderização mais rápida que o computador.

In [ ]:
# 1) Instalar FFmpeg e dependências
!apt-get -qq update
!apt-get -qq install -y ffmpeg curl
!pip -q install fastapi uvicorn python-multipart requests
!ffmpeg -version | head -n 1


In [ ]:
# 2) Instalar Cloudflare Tunnel (URL HTTPS temporária)
!curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version


In [ ]:
# 3) Baixar do GitHub a mesma API utilizada pelo site e validar a sintaxe
!curl -fsSL https://raw.githubusercontent.com/manhwavaz1-lab/manhwa-recreator-studio/main/colab/manhwa_cpu_api.py -o /content/manhwa_cpu_api.py
!python -m py_compile /content/manhwa_cpu_api.py
print('API baixada e validada.')


In [ ]:
# 4) Iniciar API e túnel. Mantenha esta célula em execução.
import os, re, secrets, subprocess, time, threading, queue, requests
from IPython.display import display, Markdown

# Encerrar processos antigos caso esta célula seja executada novamente.
for process_name in ('api_process', 'tunnel_process'):
    old = globals().get(process_name)
    if old is not None and old.poll() is None:
        old.terminate()

RENDER_TOKEN = secrets.token_urlsafe(32)
os.environ['MANHWA_RENDER_TOKEN'] = RENDER_TOKEN
api_process = subprocess.Popen(
    ['uvicorn', 'manhwa_cpu_api:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd='/content', stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT
)

# Aguardar a API ficar pronta antes de abrir o túnel.
api_ready = False
for _ in range(60):
    if api_process.poll() is not None:
        raise RuntimeError('A API encerrou durante a inicialização. Reexecute a célula da API e confira as dependências.')
    try:
        check = requests.get('http://127.0.0.1:8000/health', timeout=2)
        if check.ok and check.json().get('status') == 'ok':
            api_ready = True
            break
    except requests.RequestException:
        time.sleep(1)
if not api_ready:
    api_process.terminate()
    raise RuntimeError('A API não respondeu em 60 segundos.')

log_queue = queue.Queue()
tunnel_process = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)
def collect_tunnel_logs():
    for line in iter(tunnel_process.stdout.readline, ''):
        log_queue.put(line)
reader_thread = threading.Thread(target=collect_tunnel_logs, daemon=True)
reader_thread.start()

PUBLIC_URL = None
deadline = time.time() + 90
while time.time() < deadline and tunnel_process.poll() is None:
    try:
        line = log_queue.get(timeout=1)
    except queue.Empty:
        continue
    found = re.search(r'https://[a-z0-9-]+\\.trycloudflare\\.com', line)
    if found:
        PUBLIC_URL = found.group(0)
        break
if not PUBLIC_URL:
    tunnel_process.terminate()
    api_process.terminate()
    raise RuntimeError('O Cloudflare Tunnel não forneceu uma URL em 90 segundos. Execute novamente esta célula.')

display(Markdown(
    '## Renderizador conectado\n\n'
    + '**URL HTTPS:** `' + PUBLIC_URL + '`\n\n'
    + '**Token desta sessão:** `' + RENDER_TOKEN + '`\n\n'
    + '**Teste:** `' + PUBLIC_URL + '/health`\n\n'
    + 'Cole a URL e o token no site e clique em **Testar conexão**. Não compartilhe o token.'
))
print('API e túnel ativos. Deixe o Colab aberto enquanto o site renderiza.')


## Como usar no site
1. Abra o [Manhwa Recreator Studio](https://manhwa-panel-remix-studio.lovable.app/).
2. Na área **Conexão Google Colab**, cole a URL HTTPS e o token mostrados acima.
3. Clique em **Testar conexão** e aguarde o status conectado.
4. Selecione o vídeo e a moldura, configure formato, resolução, zoom e posição.
5. Clique em **Criar vídeo**. O navegador envia os arquivos ao Colab e baixa o MP4 final quando a API concluir.

**Importante:** URL e token mudam em cada sessão. O teste `/health` confirma apenas que a API está acessível; a renderização depende dos recursos e do espaço temporário disponíveis no Colab. 4K e vídeos longos podem exceder os limites da sessão gratuita. A porcentagem no site acompanha transferência de dados, não o progresso interno do FFmpeg.

In [ ]:
# Opcional: executar para encerrar API e túnel
for process in (globals().get('tunnel_process'), globals().get('api_process')):
    if process is not None and process.poll() is None:
        process.terminate()
print('API e túnel encerrados.')
